# Fase 2: Modelado Relacional y Persistencia Parquet

El objetivo de este cuaderno es tomar el pipeline ETL masivo validado en la sección anterior y aplicar los principios estrictos de **Modelado Dimensional (Tercera Forma Normal - 3NF)**. 

La anomalía estructural original consistía en una relación de *Muchos a Muchos* (M:N) encapsulada dentro de una tabla transaccional plana: una misma pregunta poseía múltiples tecnologías, y de igual manera, una tecnología específica era transversal a miles de preguntas distintas. Este diseño descompone de forma lógica el `DataFrame` masivo resultante de la mutación (*explode*) en tres entidades relacionales robustas, garantizando la integridad referencial y preparando el terreno para una lectura de alta compresión y extrema velocidad.

## 1. Replicación del Pipeline ETL Base

Debido a que los cuadernos en Jupyter funcionan en entornos de memoria temporal aislados, procedemos a recrear el grafo de ejecución del pipeline y ordenar su materialización en memoria mediante el motor de streaming de Polars (`.collect(streaming=True)`).

In [5]:
import polars as pl
import time
import psutil
import os

# Funciones de profiling de rendimiento para evidenciar la eficiencia de Polars
def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024) # Retorna memoria en MB

# 1. Definición de la ruta de ingesta
file_path = '../data/datos_crudos/Questions.csv'
if not os.path.exists(file_path) and os.path.exists('../data/raw/Questions.csv'):
    file_path = '../data/raw/Questions.csv'

# 2. Recreación del Query Plan Optimizado
q = pl.scan_csv(file_path, ignore_errors=True, encoding='utf8-lossy')

q_filtered = q.filter(
    pl.col("CreationDate").str.slice(0, 4).cast(pl.Int32) >= 2015
)

#Paso 3: Disociación y Normalización Vectorial de la columna 'Tags'
q_exploded = (
    q_filtered
    # 1. BUENA PRÁCTICA: Eliminar nulos de la columna objetivo ANTES de operar
    # Alternativamente, puedes usar .with_columns(pl.col("Tags").fill_null("")) si deseas conservar la fila
    .with_columns(pl.col("Tags").fill_null(""))
    
    # 2. Rompemos la cadena
    .with_columns(
        pl.col("Tags").str.split("|")
    )
    
    # 3. Muta la cardinalidad
    .explode("Tags")
    
    # 4. SOLUCIÓN AL ERROR DEL CUADERNO: Uso de la API moderna de Polars
    .with_columns(
        pl.col("Tags").str.to_lowercase().str.strip_chars()
    )
    
    # 5. Limpieza complementaria
    .filter(
        pl.col("Tags").str.len_chars() > 0
    )
)

# Benchmark y Profiling de Ejecución del DAG (Directed Acyclic Graph)
start_time = time.time()
mem_before = get_memory_usage()

try:
    # Documentación de Volúmenes (Consultas perezosas específicas para conteo optimizado)
    # 1) Filas originales crudas
    original_rows = q.select(pl.len()).collect().item()
    print(f"1) Filas originales totales en archivo: {original_rows:,}")

    # 2) Filas posteriores al Filtrado Anticipado
    filtered_rows = q_filtered.select(pl.len()).collect().item()
    print(f"2) Filas retenidas tras filtrado temporal (>= 2015): {filtered_rows:,}")
    
    # Paso 4: Materialización Controlada final de la Normalización Vectorial
    # collect(streaming=True) procesa grandes batches excediendo capacidad RAM
    df_base = q_exploded.collect(streaming=True)
    
    # 3) Filas finales mutadas luego del Explode dimensional
    final_rows = df_base.height
    print(f"3) Filas totales luego de la disociación vectorial (explode): {final_rows:,}")

    # Profiling Final
    mem_after = get_memory_usage()
    end_time = time.time()
    
    print("\n--- Métricas de Profiling (Polars vs Pandas) ---")
    print(f"Tiempo de Ejecución Total del Pipeline: {end_time - start_time:.2f} segundos")
    print(f"Consumo Neto de Memoria RAM durante ejecución: {max(0, mem_after - mem_before):.2f} MB")
    print("Nota: Un pipeline Pandas equivalente habría generado un Out-Of-Memory (OOM) o excedido el límite aceptable de ejecución.")
    
except Exception as e:
    print(f"Nota Operacional: No se pudo materializar el grafo de ejecución. Asegúrese de que el archivo 'Questions.csv' esté descargado en el path correcto. Error: {e}")

1) Filas originales totales en archivo: 16,055,694
2) Filas retenidas tras filtrado temporal (>= 2015): 16,055,694


C:\Users\PC MASTER\AppData\Local\Temp\ipykernel_14520\161097100.py:65: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_base = q_exploded.collect(streaming=True)


3) Filas totales luego de la disociación vectorial (explode): 48,050,131

--- Métricas de Profiling (Polars vs Pandas) ---
Tiempo de Ejecución Total del Pipeline: 4.09 segundos
Consumo Neto de Memoria RAM durante ejecución: 7913.36 MB
Nota: Un pipeline Pandas equivalente habría generado un Out-Of-Memory (OOM) o excedido el límite aceptable de ejecución.


## 2. Construcción del Modelo Relacional Normalizado

En este punto, el conjunto de datos cargado contiene redundancia metadato-intensiva (por ejemplo, una pregunta con 5 etiquetas multiplica su título, puntaje y fecha 5 veces de forma contigua en memoria). 

Desarmaremos lógicamente este dataframe `df_base` conformando el siguiente esquema estrella:
1. **`dim_questions` (Entidad Maestra):** Dimensión conteniendo el acervo de preguntas y suprimiendo la redundancia de datos por la clave primaria `Id`.
2. **`dim_tags` (Entidad de Inventario):** Catálogo referencial con los valores únicos de tecnologías.
3. **`fact_question_tags` (Tabla Puente de Hechos):** Estructura esbelta puramente transaccional.

In [8]:
print("Procesando las extracciones dimensionales...")

# ---------------------------------------------------------
# Entidad Maestra: dim_questions
# ---------------------------------------------------------
# Seleccionamos los atributos intrínsecos junto con ClosedDate.
# Garantizamos unicidad, derivamos la variable booleana analítica y luego
# desechamos el string original de ClosedDate para ahorrar memoria RAM.
dim_questions = (
    df_base.select(["Id", "CreationDate", "Title", "Score", "ClosedDate"])
    .unique(subset=["Id"])
    .with_columns(
        pl.col("ClosedDate").is_not_null().alias("es_cerrada")
    )
    .drop("ClosedDate")
)

# ---------------------------------------------------------
# Entidad de Inventario: dim_tags
# ---------------------------------------------------------
# Filtramos todo el vector de etiquetas extraído para conformar nuestro maestro de tecnologías.
dim_tags = (
    df_base.select(["Tags"])
    .unique()
)

# ---------------------------------------------------------
# Tabla Puente Transaccional: fact_question_tags
# ---------------------------------------------------------
# Aislamos el núcleo asociativo de cruce. Esta tabla es la más alta en cardinalidad.
fact_question_tags = (
    df_base.select(["Id", "Tags"])
)

print("Descomposición lógica relacional completada.")
print(f"-> dim_questions: {dim_questions.height:,} preguntas únicas (incluye variable 'es_cerrada')")
print(f"-> dim_tags: {dim_tags.height:,} tecnologías únicas de catálogo")
print(f"-> fact_question_tags: {fact_question_tags.height:,} tuplas asociativas M:N")

Procesando las extracciones dimensionales...
Descomposición lógica relacional completada.
-> dim_questions: 16,055,694 preguntas únicas (incluye variable 'es_cerrada')
-> dim_tags: 63,508 tecnologías únicas de catálogo
-> fact_question_tags: 48,050,131 tuplas asociativas M:N


### Racionalidad de la Tabla Puente Transaccional

La tabla `fact_question_tags` ha neutralizado la ambigua relación fundacional M:N y la ha transformado estructuralmente en relaciones asimétricas *Uno a Muchos* (1:M). Al cruzar esta tabla esbelta en la fase analítica (por ejemplo, para generar conteos de *stack tecnológico* interanuales), dispondremos de la **columna vertebral para las agregaciones** del dashboard minimizando la huella en memoria.

## 3. Arquitectura de Entidad-Relación y Persistencia (ERD)
```mermaid
erDiagram
    dim_questions ||--o{ fact_question_tags : "tiene (1:M)"
    dim_tags ||--o{ fact_question_tags : "categoriza (1:M)"

    dim_questions {
        Int64 Id PK
        String CreationDate
        String Title
        Int64 Score
        Boolean es_cerrada
    }

    dim_tags {
        String Tags PK
    }

    fact_question_tags {
        Int64 Id FK
        String Tags FK
    }

### Ventajas Estratégicas de la Persistencia en Parquet para la API

Para materializar permanentemente este logro estructural, exportaremos las tres entidades utilizando **Apache Parquet**. 

Esta decisión no es aleatoria. Para la arquitectura *backend* de **FastAPI** encargada de disponibilizar los endpoints del dashboard en la siguiente fase, leer directamente desde los archivos `.parquet` presenta beneficios de alto impacto:
1. **Alta Compresión Física:** Parquet implementa diccionarios y codificación columnar nativa, logrando disminuir el almacenamiento requerido en disco a magnitudes ínfimas frente al archivo plano CSV.
2. **Proyección (Column Pruning):** FastAPI con Polars podrá escanear del disco estrictamente las columnas requeridas (e.g. `Id` y `CreationDate`) para las consultas analíticas de los usuarios, evadiendo cargar metadatos innecesarios a la RAM.
3. **Preservación Inmutable de Tipos (Schema Enforcing):** A diferencia de `.csv` o `.json`, Parquet empaqueta el esquema fuertemente tipado en los metadatos internos del archivo. De esta forma, el motor evita incurrir en costosos procesos de reinferencia algorítmica de tipos textuales.

In [9]:
# ---------------------------------------------------------
# Exportación Persistente en Apache Parquet
# ---------------------------------------------------------

# 1. Definición y creación segura del subdirectorio procesado
output_dir = '../data/datos_procesados/'
os.makedirs(output_dir, exist_ok=True)

print("Iniciando escritura en formato de alta densidad Apache Parquet...")

# 2. Ejecución I/O
dim_questions.write_parquet(os.path.join(output_dir, 'dim_questions.parquet'))
dim_tags.write_parquet(os.path.join(output_dir, 'dim_tags.parquet'))
fact_question_tags.write_parquet(os.path.join(output_dir, 'fact_question_tags.parquet'))

print("Migración y persistencia estructural completada de forma segura.")
print(f"-> Artefactos de datos inmutables generados en el volumen: {os.path.abspath(output_dir)}")

Iniciando escritura en formato de alta densidad Apache Parquet...
Migración y persistencia estructural completada de forma segura.
-> Artefactos de datos inmutables generados en el volumen: c:\proyecto_de_grado\Pulso-Tecnologico-Stack-Overflow-Dashboard\pulso-tecnologico\data\datos_procesados
